In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import torch
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
from torch.utils.data import Dataset, DataLoader

nltk.download('punkt')
nltk.download('punkt_tab')


### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [ ]:
### loading clean text

df = pd.read_csv("./data/clean/fake_job_postings_nlp.csv")

df.head()



In [ ]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [ ]:
X_train = train_data['full_text']
y_train = train_data['fraudulent']

X_val = val_data['full_text']
y_val = val_data['fraudulent']

X_test = test_data['full_text']
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

In [ ]:
# Tokenize
def tokenize(text):
    return word_tokenize(text.lower())

# Build vocab from training data only
counter = Counter()
for text in X_train:
    counter.update(tokenize(text))

vocab = {"<unk>": 0, "<pad>": 1}
for word, freq in counter.items():
    if freq >= 1:   # raise to e.g. 2 to filter rare words
        vocab[word] = len(vocab)

# Convert text to integer sequences
def text_pipeline(text):
    return [vocab.get(token, vocab["<unk>"]) for token in tokenize(text)]

X_train_tok = [text_pipeline(text) for text in X_train]
X_val_tok   = [text_pipeline(text) for text in X_val]
X_test_tok  = [text_pipeline(text) for text in X_test]

print(f"Vocab size: {len(vocab)}")
print(f"Example: {X_train[0]}")
print(f"Tokenized: {X_train_tok[0]}")

### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens